In [229]:
import pandas as pd
# import fireducks.pandas as pd
import numpy as np
LOWER_T_VALUE = 6*2
LAT_CELLS = 300
LON_CELLS = 300

LAT_RANGE = 300 - 0
LON_RANGE = 300 - 0

parquet = '../output/models/custom_ds_latent_size_128a_fivo_adamw/logprob.parquet'
def get_bin_map():
    _ds = pd.read_parquet(parquet, columns=['log_prob'])
    lat_lon_ds = pd.read_parquet('../new_data/ais_test_by_bins.parquet', columns=['latitude', 'longitude'])
    _ds = pd.merge(_ds, lat_lon_ds, how='inner', left_index=True, right_index=True)
    _ds = _ds.reset_index()
    _ds = _ds.set_index('latitude')
    _ds = _ds.set_index('longitude', append=True)
    # higher than a timestamp
    _ds = _ds[_ds['t']>= LOWER_T_VALUE]
    _ds = _ds.drop(['t', 'track_id'], axis=1)
    quantile=1.64
    _ds = _ds[_ds.groupby(level=[0,1], sort=True)['log_prob'].transform(lambda ds: ( ds - ds.mean()) <= quantile * ds.std())]
    _ds = _ds.sort_index()
    return _ds

def get_test_set():
    _ds = pd.read_parquet(parquet, columns=['log_prob'])
    lat_lon_ds = pd.read_parquet('../new_data/ais_test_by_bins.parquet', columns=['latitude', 'longitude'])
    _ds = pd.merge(_ds, lat_lon_ds, how='inner', left_index=True, right_index=True)
    return _ds


input = get_test_set()
input = input.reset_index()
input = input.set_index('latitude')
input = input.set_index('longitude', append=True)
binmap = get_bin_map()
binmap = binmap.rename({'log_prob':'sample'}, axis=1)

In [230]:
from scipy import stats
import numpy as np


def eval_cdf(sample, value):
    return stats.gaussian_kde(sample).integrate_box_1d(-np.inf, value)

# TODO: something in the join fails i think
binmap_dist = binmap.groupby(level=[0,1]).agg(dist_sample=('sample',lambda sample: sample.to_list()))
input = pd.merge(input, binmap_dist, how='inner', left_index=True, right_index=True)
input['anomaly'] = (input.apply(lambda row: eval_cdf(row['dist_sample'], row['log_prob']), axis=1) < 0.1)

In [231]:
input = input.reset_index()
input = input.set_index('track_id')
input = input.set_index('t', append=True)
input = input.sort_index()

# TODO: fix the source of this
input = input.groupby(level=[0,1]).first()
input

latitude  longitude   log_prob  \
track_id t                                    
239      10       167         73 -59.682678   
         18       166         62 -56.361076   
         19       166         61 -55.343349   
         25       165         53 -53.193989   
         26       165         53 -48.291996   
...               ...        ...        ...   
65532    30       227        120 -16.252213   
         31       227        121 -15.667435   
         32       228        122 -13.182238   
         33       229        123 -16.000687   
         34       230        124 -13.060799   

                                                   dist_sample  anomaly  
track_id t                                                               
239      10         [-13.481100082397461, -51.535377502441406]    False  
         18            [-56.36107635498047, -18.8808536529541]    False  
         19  [-55.34334945678711, -11.381169319152832, -28....    False  
         25  [-53.19398880004883, -48.291996002197266, -44....     True  
         26  [-53.19398880004883, -48.291996002197266, -44....     True  
...                                                        ...      ...  
65532    30  [-12.59228515625, -12.405372619628906, -10.700...    False  
         31  [-24.76830291748047, -22.076587677001953, -13....    False  
         32  [-14.525707244873047, -13.241288185119629, -12...    False  
         33  [-16.11602210998535, -17.5559024810791, -30.29...    False  
         34  [-19.08651351928711, -15.131471633911133, -12....    False  

[78506 rows x 5 columns]

In [232]:
input.iloc[input.index.get_level_values(1)>11]

latitude  longitude   log_prob  \
track_id t                                    
239      18       166         62 -56.361076   
         19       166         61 -55.343349   
         25       165         53 -53.193989   
         26       165         53 -48.291996   
         27       165         53 -44.429600   
...               ...        ...        ...   
65532    30       227        120 -16.252213   
         31       227        121 -15.667435   
         32       228        122 -13.182238   
         33       229        123 -16.000687   
         34       230        124 -13.060799   

                                                   dist_sample  anomaly  
track_id t                                                               
239      18            [-56.36107635498047, -18.8808536529541]    False  
         19  [-55.34334945678711, -11.381169319152832, -28....    False  
         25  [-53.19398880004883, -48.291996002197266, -44....     True  
         26  [-53.19398880004883, -48.291996002197266, -44....     True  
         27  [-53.19398880004883, -48.291996002197266, -44....     True  
...                                                        ...      ...  
65532    30  [-12.59228515625, -12.405372619628906, -10.700...    False  
         31  [-24.76830291748047, -22.076587677001953, -13....    False  
         32  [-14.525707244873047, -13.241288185119629, -12...    False  
         33  [-16.11602210998535, -17.5559024810791, -30.29...    False  
         34  [-19.08651351928711, -15.131471633911133, -12....    False  

[62256 rows x 5 columns]

In [233]:
from functools import reduce
def nCr_old(n, r):
    """Function calculates the number of combinations (n choose r)"""
    r = min(r, n-r)
    numer = reduce(op.mul, range(n, n-r, -1), 1)
    denom = reduce(op.mul, range(1, r+1), 1)
    return numer//denom


def NFA_old(ns,k):
    """Number of False Alarms"""
    B = 0
    for t in range(k,ns+1):
        B += nCr_old(ns,t)*(0.1**t)*(0.9**(ns-t))
    return 300*B


def contrario_detection_old(v_A_ : np.ndarray[bool],epsilon=0.0091):
    """
    A contrario detection algorithms
    INPUT:
        v_A_: abnormal point indicator vector
        epsilon: threshold
    OUTPUT:
        v_anomalies: abnormal segment indicator vector

    """
    v_anomalies = np.zeros(len(v_A_), dtype=bool)
    max_seq_len = min(MAX_SEQUENCE_LENGTH, len(v_A_))
    for d_ns in range(max_seq_len,0,-1):
        for d_ci in range(max_seq_len+1-d_ns):
            v_xi = v_A_[d_ci:d_ci+d_ns]
            d_k_xi = int(np.count_nonzero(v_xi))
            if NFA_old(d_ns,d_k_xi)<epsilon:
                v_anomalies[d_ci:d_ci+d_ns] = True
    return v_anomalies


a = []
def f(x):
    return x.any()

def g(x):
    x = x.iloc[::-1].rolling(window=24, min_periods=0).apply(f).iloc[::-1]
    return x.any()

CONTRARIO_EPS = 1e-9
def apply_windowed_a_contrario(v_A):
    v_anomalies = np.zeros(len(v_A), dtype=bool)
    for d_i_4h in range(0,len(v_A)+1-24):
        v_A_4h = v_A[d_i_4h:d_i_4h+24]
        v_anomalies_i = contrario_detection_old(v_A_4h,CONTRARIO_EPS)
        v_anomalies[d_i_4h:d_i_4h+24][v_anomalies_i] = True
    return v_anomalies


In [248]:
sliding_window_view(input['anomaly'],3)

array([[False, False, False],
       [False, False,  True],
       [False,  True,  True],
       ...,
       [False, False, False],
       [False, False, False],
       [False, False, False]])

In [284]:
pd.Series(list(sliding_window_view(input['anomaly'],24)), index=input.index[:-24+1]).map(contrario_detection_old)

KeyboardInterrupt: 

In [285]:
from numba import bool_, guvectorize

@guvectorize([(bool_[:], bool_[:])], '(n)->(n)')
def a_contrario_detection(v_A_, v_anomalies):
    epsilon=0.0091
    max_seq_len = min(MAX_SEQUENCE_LENGTH, len(v_A_))
    for d_ns in range(max_seq_len,0,-1):
        for d_ci in range(max_seq_len+1-d_ns):
            v_xi = v_A_[d_ci:d_ci+d_ns]
            d_k_xi = int(np.count_nonzero(v_xi))
            if NFA_old(d_ns,d_k_xi)<epsilon:
                v_anomalies[d_ci:d_ci+d_ns] = True
    return v_anomalies


def contrario_detection_old(v_A_ : np.ndarray[bool],epsilon=0.0091):
    """
    A contrario detection algorithms
    INPUT:
        v_A_: abnormal point indicator vector
        epsilon: threshold
    OUTPUT:
        v_anomalies: abnormal segment indicator vector

    """
    v_anomalies = np.zeros(len(v_A_), dtype=np.bool_)
    max_seq_len = min(MAX_SEQUENCE_LENGTH, len(v_A_))
    for d_ns in range(max_seq_len,0,-1):
        for d_ci in range(max_seq_len+1-d_ns):
            v_xi = v_A_[d_ci:d_ci+d_ns]
            d_k_xi = int(np.count_nonzero(v_xi))
            if NFA_old(d_ns,d_k_xi)<epsilon:
                v_anomalies[d_ci:d_ci+d_ns] = True
    return v_anomalies



def apply_windowed_a_contrario(v_A):
    v_anomalies = np.zeros(len(v_A), dtype=np.bool_)
    for d_i_4h in range(0,len(v_A)+1-24):
        v_A_4h = v_A[d_i_4h:d_i_4h+24]
        v_anomalies_i = contrario_detection_old(v_A_4h,CONTRARIO_EPS)
        v_anomalies[d_i_4h:d_i_4h+24][v_anomalies_i] = True
    return v_anomalies

TypingError: Failed in nopython mode pipeline (step: nopython frontend)
Untyped global name 'NFA_old': Cannot determine Numba type of <class 'function'>

File "../../../../../../../../tmp/ipykernel_2777578/4143952689.py", line 11:
<source missing, REPL/exec in use?>

During: Pass nopython_type_inference

In [239]:
from numpy.lib.stride_tricks import sliding_window_view

In [245]:
sliding_window_view(np.arange(6), 7)

ValueError: window shape cannot be larger than input array shape

In [236]:
from numba import guvectorize

In [ ]:
input['anomaly'].apply(lambda col)

track_id
239       True
299       True
332       True
358      False
404       True
         ...  
65485     True
65493     True
65503     True
65509     True
65532     True
Name: outlier, Length: 1851, dtype: bool

In [ ]:
def apply(x):
    return 

for window in input.groupby(by='track_id')['outlier'].rolling(window=24):
    a = window

def f(x):
    return x.any()
a.iloc[::-1].rolling(window=24, min_periods=0).apply(f).iloc[::-1]